# Phase 3 — Baseline Models

## Objective

This notebook implements baseline models for Twitter bot detection using the frozen preprocessing pipeline from Phase 2.

The goal is to establish reference performance for:

1. Tabular-only model
2. Text-only model using XLM-R
3. Graph-based model (optional)

All experiments use the frozen Train/Validation/Test splits without any additional preprocessing or data leakage.

In [4]:
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report
)

print("Libraries imported successfully")

Libraries imported successfully


In [5]:
# Project root
PROJECT_ROOT = Path(
    r"C:\Users\p.vazifeh\Desktop\University\Project\Bot Detection Implementation"
)

FINAL_DATA = PROJECT_ROOT / "final_data"
SPLITS_DIR = FINAL_DATA / "splits"

TABULAR_DIR = SPLITS_DIR / "tabular"
TEXT_DIR = SPLITS_DIR / "text"
GRAPH_DIR = SPLITS_DIR / "graph"

CONFIG_DIR = FINAL_DATA / "config"

print("Project root:", PROJECT_ROOT)
print("Final data:", FINAL_DATA)

print("\nExisting folders:")
for p in [TABULAR_DIR, TEXT_DIR, GRAPH_DIR, CONFIG_DIR]:
    print(p, "->", p.exists())

Project root: C:\Users\p.vazifeh\Desktop\University\Project\Bot Detection Implementation
Final data: C:\Users\p.vazifeh\Desktop\University\Project\Bot Detection Implementation\final_data

Existing folders:
C:\Users\p.vazifeh\Desktop\University\Project\Bot Detection Implementation\final_data\splits\tabular -> True
C:\Users\p.vazifeh\Desktop\University\Project\Bot Detection Implementation\final_data\splits\text -> True
C:\Users\p.vazifeh\Desktop\University\Project\Bot Detection Implementation\final_data\splits\graph -> True
C:\Users\p.vazifeh\Desktop\University\Project\Bot Detection Implementation\final_data\config -> True


In [6]:
# Load frozen tabular datasets

train_tabular = pd.read_csv(TABULAR_DIR / "train_tabular.csv")
val_tabular = pd.read_csv(TABULAR_DIR / "validation_tabular.csv")
test_tabular = pd.read_csv(TABULAR_DIR / "test_tabular.csv")


print("="*90)
print("FROZEN TABULAR DATA")
print("="*90)

print("Train:")
print(train_tabular.shape)

print("\nValidation:")
print(val_tabular.shape)

print("\nTest:")
print(test_tabular.shape)


display(train_tabular.head())

FROZEN TABULAR DATA
Train:
(671, 43)

Validation:
(144, 43)

Test:
(144, 43)


,screen_name,user_key,label_binary,followers_count,friends_count,favourites_count,listed_count,media_count,statuses_count,user_age,...,unique_mention_rate_per_tweet,mean_user_mentions_per_tweet,retweet_as_tweet_rate,no_retweet_tweets,mean_retweets_per_tweet,description_length,followers_friend_ratio,num_digits_in_name,num_digits_in_username,url_in_description
0,Dokhi_sag,dokhi_sag,0,2.260710,1.922168,-0.717008,0.800936,-0.218288,-0.648278,-0.393013,...,-0.257604,-0.278501,0.047134,-0.648059,0.372444,0.478261,-0.344915,0.0,0.00000,0.0
1,History_Wall,history_wall,0,-0.661092,-0.412186,-0.211043,-0.671950,-1.161989,-0.956884,-0.245569,...,0.406515,0.395583,-0.608860,-0.958021,-0.010403,0.565217,-0.038019,0.0,0.00000,0.0
2,corrrdeliiiaaa,corrrdeliiiaaa,0,-0.974405,-1.492217,1.227229,0.000000,1.007042,1.017413,-0.140252,...,0.742396,0.273356,0.897650,1.017039,-1.614247,-0.663043,0.685375,0.0,0.00000,0.0
3,ahmad_313m,ahmad_313m,0,-0.103043,-2.765518,-2.411713,0.714903,-0.881804,-1.224861,-0.600565,...,-0.257604,-0.278501,-0.032728,-1.224341,-1.024064,-0.663043,2.410511,0.0,1.26186,0.0
4,kouresh_kabir,kouresh_kabir,0,0.070104,0.592806,-0.147311,-0.199064,0.668362,-0.182317,0.578988,...,0.355769,0.335443,-0.875661,-0.185225,0.268458,0.771739,-0.375985,0.0,0.00000,0.0


In [7]:
# Separate features and labels

TARGET = "label_binary"

DROP_COLUMNS = [
    "screen_name",
    "user_key",
    TARGET
]


X_train = train_tabular.drop(columns=DROP_COLUMNS)
y_train = train_tabular[TARGET]

X_val = val_tabular.drop(columns=DROP_COLUMNS)
y_val = val_tabular[TARGET]

X_test = test_tabular.drop(columns=DROP_COLUMNS)
y_test = test_tabular[TARGET]


print("="*90)
print("TABULAR MATRICES")
print("="*90)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print()

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print()

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)


print("\nLabel distribution")

print("\nTrain:")
print(y_train.value_counts())

print("\nValidation:")
print(y_val.value_counts())

print("\nTest:")
print(y_test.value_counts())

TABULAR MATRICES
X_train: (671, 40)
y_train: (671,)

X_val: (144, 40)
y_val: (144,)

X_test: (144, 40)
y_test: (144,)

Label distribution

Train:
label_binary
0    540
1    131
Name: count, dtype: int64

Validation:
label_binary
0    116
1     28
Name: count, dtype: int64

Test:
label_binary
0    116
1     28
Name: count, dtype: int64


In [8]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)


def evaluate_model(name, y_true, y_pred, y_prob):
    
    print("="*90)
    print(name)
    print("="*90)

    results = {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_true, y_prob),
        "PR_AUC": average_precision_score(y_true, y_prob)
    }

    for k, v in results.items():
        if k != "Model":
            print(f"{k}: {v:.4f}")

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

    print("\nClassification Report:")
    print(classification_report(
        y_true,
        y_pred,
        target_names=["Human", "Bot"],
        zero_division=0
    ))

    return results

In [9]:
from sklearn.linear_model import LogisticRegression


# Initialize model
logreg = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)


# Train
logreg.fit(
    X_train,
    y_train
)


print("Logistic Regression trained successfully")

Logistic Regression trained successfully


In [10]:
# Predictions

val_prob_logreg = logreg.predict_proba(X_val)[:, 1]
val_pred_logreg = logreg.predict(X_val)


test_prob_logreg = logreg.predict_proba(X_test)[:, 1]
test_pred_logreg = logreg.predict(X_test)


# Evaluate

logreg_val_results = evaluate_model(
    "Logistic Regression - Validation",
    y_val,
    val_pred_logreg,
    val_prob_logreg
)


logreg_test_results = evaluate_model(
    "Logistic Regression - Test",
    y_test,
    test_pred_logreg,
    test_prob_logreg
)

Logistic Regression - Validation
Accuracy: 0.6667
Precision: 0.3387
Recall: 0.7500
F1: 0.4667
ROC_AUC: 0.7321
PR_AUC: 0.4462

Confusion Matrix:
[[75 41]
 [ 7 21]]

Classification Report:
              precision    recall  f1-score   support

       Human       0.91      0.65      0.76       116
         Bot       0.34      0.75      0.47        28

    accuracy                           0.67       144
   macro avg       0.63      0.70      0.61       144
weighted avg       0.80      0.67      0.70       144

Logistic Regression - Test
Accuracy: 0.7083
Precision: 0.3600
Recall: 0.6429
F1: 0.4615
ROC_AUC: 0.7722
PR_AUC: 0.4946

Confusion Matrix:
[[84 32]
 [10 18]]

Classification Report:
              precision    recall  f1-score   support

       Human       0.89      0.72      0.80       116
         Bot       0.36      0.64      0.46        28

    accuracy                           0.71       144
   macro avg       0.63      0.68      0.63       144
weighted avg       0.79      0.71

In [11]:
import os
import pandas as pd


RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)


baseline_results = pd.DataFrame([
    logreg_val_results,
    logreg_test_results
])


baseline_results.to_csv(
    RESULTS_DIR / "baseline_results.csv",
    index=False
)


print("Baseline results saved:")
print(RESULTS_DIR / "baseline_results.csv")

display(baseline_results)

Baseline results saved:
C:\Users\p.vazifeh\Desktop\University\Project\Bot Detection Implementation\results\baseline_results.csv


,Model,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC
0,Logistic Regression - Validation,0.666667,0.33871,0.750000,0.466667,0.732143,0.446154
1,Logistic Regression - Test,0.708333,0.36000,0.642857,0.461538,0.772167,0.494570


In [12]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader


print("PyTorch version:", torch.__version__)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)

PyTorch version: 2.14.0+cpu
Device: cpu


In [13]:
# Convert tabular data to PyTorch tensors

X_train_tensor = torch.tensor(
    X_train.values,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train.values,
    dtype=torch.float32
).unsqueeze(1)


X_val_tensor = torch.tensor(
    X_val.values,
    dtype=torch.float32
)

y_val_tensor = torch.tensor(
    y_val.values,
    dtype=torch.float32
).unsqueeze(1)


X_test_tensor = torch.tensor(
    X_test.values,
    dtype=torch.float32
)

y_test_tensor = torch.tensor(
    y_test.values,
    dtype=torch.float32
).unsqueeze(1)



# Create datasets

train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

val_dataset = TensorDataset(
    X_val_tensor,
    y_val_tensor
)

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)



# DataLoaders

BATCH_SIZE = 32


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)



print("="*90)
print("PYTORCH TABULAR DATA")
print("="*90)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

print("\nTensor shapes:")
print("X_train:", X_train_tensor.shape)
print("y_train:", y_train_tensor.shape)

PYTORCH TABULAR DATA
Train batches: 21
Validation batches: 5
Test batches: 5

Tensor shapes:
X_train: torch.Size([671, 40])
y_train: torch.Size([671, 1])


In [14]:
class TabularMLP(nn.Module):
    
    def __init__(self, input_dim):
        super().__init__()
        
        self.network = nn.Sequential(
            
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(32, 1)
        )
    
    
    def forward(self, x):
        return self.network(x)



# Initialize model

mlp_model = TabularMLP(
    input_dim=X_train.shape[1]
).to(DEVICE)



print(mlp_model)

TabularMLP(
  (network): Sequential(
    (0): Linear(in_features=40, out_features=64, bias=True)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=64, out_features=32, bias=True)
    (5): ReLU()
    (6): Dropout(p=0.3, inplace=False)
    (7): Linear(in_features=32, out_features=1, bias=True)
  )
)


In [15]:
# Loss with class imbalance handling

positive_weight = (
    len(y_train[y_train == 0]) /
    len(y_train[y_train == 1])
)


pos_weight = torch.tensor(
    [positive_weight],
    dtype=torch.float32
).to(DEVICE)


criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)


optimizer = torch.optim.AdamW(
    mlp_model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)


EPOCHS = 50


print("="*90)
print("TRAINING CONFIGURATION")
print("="*90)

print("Positive class weight:", positive_weight)
print("Optimizer: AdamW")
print("Learning rate: 1e-3")
print("Epochs:", EPOCHS)
print("Loss: BCEWithLogitsLoss")

TRAINING CONFIGURATION
Positive class weight: 4.122137404580153
Optimizer: AdamW
Learning rate: 1e-3
Epochs: 50
Loss: BCEWithLogitsLoss


In [16]:
from tqdm.auto import tqdm


train_losses = []
val_losses = []


for epoch in range(EPOCHS):

    # --------------------
    # Training
    # --------------------
    mlp_model.train()

    total_train_loss = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        optimizer.zero_grad()

        logits = mlp_model(X_batch)

        loss = criterion(
            logits,
            y_batch
        )

        loss.backward()

        optimizer.step()

        total_train_loss += loss.item()


    avg_train_loss = (
        total_train_loss /
        len(train_loader)
    )


    # --------------------
    # Validation
    # --------------------
    mlp_model.eval()

    total_val_loss = 0

    with torch.no_grad():

        for X_batch, y_batch in val_loader:

            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            logits = mlp_model(X_batch)

            loss = criterion(
                logits,
                y_batch
            )

            total_val_loss += loss.item()


    avg_val_loss = (
        total_val_loss /
        len(val_loader)
    )


    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)


    if (epoch + 1) % 5 == 0:

        print(
            f"Epoch [{epoch+1}/{EPOCHS}] "
            f"Train Loss: {avg_train_loss:.4f} "
            f"Val Loss: {avg_val_loss:.4f}"
        )


print("\nTraining completed")

Epoch [5/50] Train Loss: 0.9417 Val Loss: 0.9727
Epoch [10/50] Train Loss: 0.8615 Val Loss: 0.9511
Epoch [15/50] Train Loss: 0.7769 Val Loss: 0.9729
Epoch [20/50] Train Loss: 0.7569 Val Loss: 1.0327
Epoch [25/50] Train Loss: 0.6754 Val Loss: 1.0858
Epoch [30/50] Train Loss: 0.6441 Val Loss: 1.1313
Epoch [35/50] Train Loss: 0.6258 Val Loss: 1.1536
Epoch [40/50] Train Loss: 0.5583 Val Loss: 1.2607
Epoch [45/50] Train Loss: 0.5960 Val Loss: 1.3045
Epoch [50/50] Train Loss: 0.5722 Val Loss: 1.2580

Training completed


In [17]:
from scipy.special import expit


def predict_mlp(model, loader):
    
    model.eval()
    
    probabilities = []
    labels = []
    
    with torch.no_grad():

        for X_batch, y_batch in loader:

            X_batch = X_batch.to(DEVICE)

            logits = model(X_batch)

            probs = torch.sigmoid(logits)

            probabilities.extend(
                probs.cpu().numpy().flatten()
            )

            labels.extend(
                y_batch.numpy().flatten()
            )

    return (
        np.array(labels),
        np.array(probabilities)
    )



# Validation prediction

y_val_mlp, val_prob_mlp = predict_mlp(
    mlp_model,
    val_loader
)

val_pred_mlp = (
    val_prob_mlp >= 0.5
).astype(int)



# Test prediction

y_test_mlp, test_prob_mlp = predict_mlp(
    mlp_model,
    test_loader
)

test_pred_mlp = (
    test_prob_mlp >= 0.5
).astype(int)



print("Prediction completed")

print("Validation probabilities:", val_prob_mlp.shape)
print("Test probabilities:", test_prob_mlp.shape)

Prediction completed
Validation probabilities: (144,)
Test probabilities: (144,)


In [49]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)


def evaluate_predictions(
    y_true,
    y_prob,
    split_name
):

    y_pred = (y_prob >= 0.5).astype(int)

    print("=" * 90)
    print(f"{split_name}")
    print("=" * 90)

    print(
        f"Accuracy:  {accuracy_score(y_true, y_pred):.4f}"
    )

    print(
        f"Precision: {precision_score(y_true, y_pred):.4f}"
    )

    print(
        f"Recall:    {recall_score(y_true, y_pred):.4f}"
    )

    print(
        f"F1:        {f1_score(y_true, y_pred):.4f}"
    )

    print(
        f"ROC_AUC:   {roc_auc_score(y_true, y_prob):.4f}"
    )

    print(
        f"PR_AUC:    {average_precision_score(y_true, y_prob):.4f}"
    )


    print("\nConfusion Matrix:")
    print(
        confusion_matrix(
            y_true,
            y_pred
        )
    )


    print("\nClassification Report:")
    print(
        classification_report(
            y_true,
            y_pred,
            target_names=[
                "Human",
                "Bot"
            ]
        )
    )


    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred),
        "ROC_AUC": roc_auc_score(y_true, y_prob),
        "PR_AUC": average_precision_score(y_true, y_prob)
    }



mlp_val_results = evaluate_predictions(
    y_val_mlp,
    val_prob_mlp,
    "Validation"
)


mlp_test_results = evaluate_predictions(
    y_test_mlp,
    test_prob_mlp,
    "Test"
)

Validation
Accuracy:  0.7222
Precision: 0.3750
Recall:    0.6429
F1:        0.4737
ROC_AUC:   0.7472
PR_AUC:    0.5006

Confusion Matrix:
[[86 30]
 [10 18]]

Classification Report:
              precision    recall  f1-score   support

       Human       0.90      0.74      0.81       116
         Bot       0.38      0.64      0.47        28

    accuracy                           0.72       144
   macro avg       0.64      0.69      0.64       144
weighted avg       0.79      0.72      0.75       144

Test
Accuracy:  0.6944
Precision: 0.3000
Recall:    0.4286
F1:        0.3529
ROC_AUC:   0.7451
PR_AUC:    0.4256

Confusion Matrix:
[[88 28]
 [16 12]]

Classification Report:
              precision    recall  f1-score   support

       Human       0.85      0.76      0.80       116
         Bot       0.30      0.43      0.35        28

    accuracy                           0.69       144
   macro avg       0.57      0.59      0.58       144
weighted avg       0.74      0.69      0.71  

In [19]:
import pandas as pd
import os


results_dir = os.path.join(
    PROJECT_ROOT,
    "results"
)

os.makedirs(
    results_dir,
    exist_ok=True
)


baseline_results = pd.DataFrame(
    [
        {
            "Model": "Logistic Regression - Validation",
            **logreg_val_results
        },
        {
            "Model": "Logistic Regression - Test",
            **logreg_test_results
        },
        {
            "Model": "Tabular MLP - Validation",
            **mlp_val_results
        },
        {
            "Model": "Tabular MLP - Test",
            **mlp_test_results
        }
    ]
)


baseline_results = baseline_results[
    [
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC_AUC",
        "PR_AUC"
    ]
]


baseline_path = os.path.join(
    results_dir,
    "baseline_results.csv"
)


baseline_results.to_csv(
    baseline_path,
    index=False
)


print("="*90)
print("BASELINE RESULTS UPDATED")
print("="*90)

print(baseline_path)

display(baseline_results)

BASELINE RESULTS UPDATED
C:\Users\p.vazifeh\Desktop\University\Project\Bot Detection Implementation\results\baseline_results.csv


,Model,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC
0,Logistic Regression - Validation,0.666667,0.33871,0.750000,0.466667,0.732143,0.446154
1,Logistic Regression - Test,0.708333,0.36000,0.642857,0.461538,0.772167,0.494570
2,Tabular MLP - Validation,0.722222,0.37500,0.642857,0.473684,0.747229,0.500635
3,Tabular MLP - Test,0.694444,0.30000,0.428571,0.352941,0.745074,0.425563


In [20]:
import json
import os


baseline_info = {
    "dataset": "Gold Binary Dataset",
    "users": 959,
    "train_users": 671,
    "validation_users": 144,
    "test_users": 144,
    "modalities": [
        "tabular",
        "text",
        "graph"
    ],
    "completed_baselines": [
        "Logistic Regression",
        "Tabular MLP"
    ]
}


path = os.path.join(
    PROJECT_ROOT,
    "results",
    "baseline_metadata.json"
)


with open(path, "w") as f:
    json.dump(
        baseline_info,
        f,
        indent=4
    )


print(path)

C:\Users\p.vazifeh\Desktop\University\Project\Bot Detection Implementation\results\baseline_metadata.json


In [21]:
# Check frozen text split directory

print("=" * 90)
print("TEXT DATA DIRECTORY")
print("=" * 90)

print(TEXT_DIR)

print("\nFiles:")

for f in os.listdir(TEXT_DIR):
    print(f)

TEXT DATA DIRECTORY
C:\Users\p.vazifeh\Desktop\University\Project\Bot Detection Implementation\final_data\splits\text

Files:
test_text.csv
text_modality_availability.csv
train_text.csv
validation_text.csv


In [22]:
# Load frozen text datasets

train_text_path = TEXT_DIR / "train_text.csv"
val_text_path = TEXT_DIR / "validation_text.csv"
test_text_path = TEXT_DIR / "test_text.csv"


train_text = pd.read_csv(train_text_path)
val_text = pd.read_csv(val_text_path)
test_text = pd.read_csv(test_text_path)


print("=" * 90)
print("FROZEN TEXT DATASETS")
print("=" * 90)

print("Train:")
print(train_text.shape)

print("\nValidation:")
print(val_text.shape)

print("\nTest:")
print(test_text.shape)


print("\nTrain columns:")
print(train_text.columns.tolist())


print("\nFirst rows:")
display(train_text.head())

FROZEN TEXT DATASETS
Train:
(12976, 6)

Validation:
(2618, 6)

Test:
(3006, 6)

Train columns:
['user_key', 'screen_name', 'published_at', 'type', 'lang', 'text_model']

First rows:


,user_key,screen_name,published_at,type,lang,text_model
0,007mi00,007mi00,2025-01-27 14:09:49+00:00,tweet,fa,سپاه جز ویرانی و دزدی و چپاول هیچ چیزی واسه ای...
1,007mi00,007mi00,2024-03-08 21:50:46+00:00,tweet,fa,میدونم کسی دیگه توییتهای منو نمیبینه ولی منم ب...
2,007mi00,007mi00,2023-09-27 19:31:12+00:00,tweet,fa,تو میایی و مقتدر ترین پادشاهی تاریخ ایران رو ر...
3,007mi00,007mi00,2023-05-17 19:03:02+00:00,tweet,fa,تنها شخصی که ایستاده او را تشویق میکنن مادر ای...
4,007mi00,007mi00,2023-04-16 21:11:48+00:00,retweet_with_comment,fa,هنوز رد هزار پای راهپیمایی قدس خشک نشده پادشاه...


In [23]:
# Attach binary labels to text datasets

label_cols = [
    "user_key",
    "label_binary"
]


train_labels = train_tabular[label_cols].copy()
val_labels = val_tabular[label_cols].copy()
test_labels = test_tabular[label_cols].copy()


train_text_labeled = train_text.merge(
    train_labels,
    on="user_key",
    how="left"
)


val_text_labeled = val_text.merge(
    val_labels,
    on="user_key",
    how="left"
)


test_text_labeled = test_text.merge(
    test_labels,
    on="user_key",
    how="left"
)


print("=" * 90)
print("TEXT LABEL JOIN AUDIT")
print("=" * 90)


for name, df in [
    ("Train", train_text_labeled),
    ("Validation", val_text_labeled),
    ("Test", test_text_labeled)
]:

    print(f"\n{name}")

    print("Rows:", len(df))

    print(
        "Missing labels:",
        df["label_binary"].isna().sum()
    )

    print(
        "Users:",
        df["user_key"].nunique()
    )

    print(
        df["label_binary"].value_counts()
    )

TEXT LABEL JOIN AUDIT

Train
Rows: 12976
Missing labels: 0
Users: 671
label_binary
0    9466
1    3510
Name: count, dtype: int64

Validation
Rows: 2618
Missing labels: 0
Users: 144
label_binary
0    1930
1     688
Name: count, dtype: int64

Test
Rows: 3006
Missing labels: 0
Users: 143
label_binary
0    2326
1     680
Name: count, dtype: int64


In [24]:
from transformers import AutoTokenizer


MODEL_NAME = "xlm-roberta-base"
MAX_LENGTH = 128


tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)


print("=" * 90)
print("XLM-R TOKENIZER")
print("=" * 90)

print("Model:", MODEL_NAME)
print("Tokenizer:", type(tokenizer).__name__)
print("Max length:", MAX_LENGTH)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

c:\Users\p.vazifeh\Desktop\University\Project\Bot Detection Implementation\env\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\p.vazifeh\.cache\huggingface\hub\models--xlm-roberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

Error while downloading from https://huggingface.co/xlm-roberta-base/resolve/main/sentencepiece.bpe.model: The read operation timed out
Trying to resume download...


tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

XLM-R TOKENIZER
Model: xlm-roberta-base
Tokenizer: XLMRobertaTokenizer
Max length: 128


In [25]:
import torch
from transformers import AutoModel


MODEL_NAME = "xlm-roberta-base"


device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)


xlmr_model = AutoModel.from_pretrained(
    MODEL_NAME
)


xlmr_model.to(device)


# Freeze parameters

for param in xlmr_model.parameters():
    param.requires_grad = False


xlmr_model.eval()


print("=" * 90)
print("XLM-R MODEL")
print("=" * 90)

print("Model:", MODEL_NAME)
print("Device:", device)

print(
    "Parameters requiring gradients:",
    sum(
        p.requires_grad
        for p in xlmr_model.parameters()
    )
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


XLM-R MODEL
Model: xlm-roberta-base
Device: cpu
Parameters requiring gradients: 0


In [26]:
# Test XLM-R forward pass

sample_text = train_text_labeled.iloc[0]["text_model"]


inputs = tokenizer(
    sample_text,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)


inputs = {
    k: v.to(device)
    for k, v in inputs.items()
}


with torch.no_grad():

    outputs = xlmr_model(
        **inputs
    )


last_hidden_state = outputs.last_hidden_state


print("=" * 90)
print("XLM-R FORWARD TEST")
print("=" * 90)

print("Input text length:", len(sample_text))

print(
    "Last hidden state shape:",
    last_hidden_state.shape
)

XLM-R FORWARD TEST
Input text length: 140
Last hidden state shape: torch.Size([1, 128, 768])


In [27]:
import torch


def mean_pooling(
    model_output,
    attention_mask
):
    
    token_embeddings = model_output.last_hidden_state
    
    input_mask_expanded = (
        attention_mask
        .unsqueeze(-1)
        .expand(token_embeddings.size())
        .float()
    )

    sum_embeddings = torch.sum(
        token_embeddings * input_mask_expanded,
        dim=1
    )

    sum_mask = torch.clamp(
        input_mask_expanded.sum(dim=1),
        min=1e-9
    )

    return sum_embeddings / sum_mask


print("Mean pooling function ready")

Mean pooling function ready


In [28]:
from torch.utils.data import DataLoader
import numpy as np


def extract_text_embeddings(
    dataframe,
    model,
    tokenizer,
    batch_size=16,
    max_length=128
):

    texts = dataframe["text_model"].tolist()

    embeddings = []

    model.eval()

    for start in range(
        0,
        len(texts),
        batch_size
    ):

        batch_texts = texts[
            start:start + batch_size
        ]


        encoded = tokenizer(
            batch_texts,
            padding="max_length",
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )


        encoded = {
            k: v.to(device)
            for k, v in encoded.items()
        }


        with torch.no_grad():

            outputs = model(
                **encoded
            )


            batch_embeddings = mean_pooling(
                outputs,
                encoded["attention_mask"]
            )


        embeddings.append(
            batch_embeddings.cpu().numpy()
        )


        if start % (batch_size * 50) == 0:
            print(
                f"Processed {min(start+batch_size, len(texts))}/{len(texts)} tweets"
            )


    return np.vstack(embeddings)



print("Embedding extraction function ready")

Embedding extraction function ready


In [29]:
# Small XLM-R embedding extraction test

sample_embeddings = extract_text_embeddings(
    train_text_labeled.head(100),
    xlmr_model,
    tokenizer,
    batch_size=16,
    max_length=128
)


print("=" * 90)
print("EMBEDDING TEST")
print("=" * 90)

print(
    "Embedding shape:",
    sample_embeddings.shape
)

print(
    "Embedding dtype:",
    sample_embeddings.dtype
)

print(
    "First embedding dimension:",
    len(sample_embeddings[0])
)

Processed 16/100 tweets
EMBEDDING TEST
Embedding shape: (100, 768)
Embedding dtype: float32
First embedding dimension: 768


In [30]:
# Extract XLM-R tweet embeddings for all splits

print("=" * 90)
print("EXTRACTING TRAIN EMBEDDINGS")
print("=" * 90)

train_tweet_embeddings = extract_text_embeddings(
    train_text_labeled,
    xlmr_model,
    tokenizer,
    batch_size=16,
    max_length=128
)


print("\nTrain embeddings shape:")
print(train_tweet_embeddings.shape)



print("\n" + "=" * 90)
print("EXTRACTING VALIDATION EMBEDDINGS")
print("=" * 90)

val_tweet_embeddings = extract_text_embeddings(
    val_text_labeled,
    xlmr_model,
    tokenizer,
    batch_size=16,
    max_length=128
)


print("\nValidation embeddings shape:")
print(val_tweet_embeddings.shape)



print("\n" + "=" * 90)
print("EXTRACTING TEST EMBEDDINGS")
print("=" * 90)

test_tweet_embeddings = extract_text_embeddings(
    test_text_labeled,
    xlmr_model,
    tokenizer,
    batch_size=16,
    max_length=128
)


print("\nTest embeddings shape:")
print(test_tweet_embeddings.shape)

EXTRACTING TRAIN EMBEDDINGS
Processed 16/12976 tweets
Processed 816/12976 tweets
Processed 1616/12976 tweets
Processed 2416/12976 tweets
Processed 3216/12976 tweets
Processed 4016/12976 tweets
Processed 4816/12976 tweets
Processed 5616/12976 tweets
Processed 6416/12976 tweets
Processed 7216/12976 tweets
Processed 8016/12976 tweets
Processed 8816/12976 tweets
Processed 9616/12976 tweets
Processed 10416/12976 tweets
Processed 11216/12976 tweets
Processed 12016/12976 tweets
Processed 12816/12976 tweets

Train embeddings shape:
(12976, 768)

EXTRACTING VALIDATION EMBEDDINGS
Processed 16/2618 tweets
Processed 816/2618 tweets
Processed 1616/2618 tweets
Processed 2416/2618 tweets

Validation embeddings shape:
(2618, 768)

EXTRACTING TEST EMBEDDINGS
Processed 16/3006 tweets
Processed 816/3006 tweets
Processed 1616/3006 tweets
Processed 2416/3006 tweets

Test embeddings shape:
(3006, 768)


In [31]:
# Save tweet embeddings

EMBEDDING_DIR = FINAL_DATA / "embeddings" / "xlmr"

EMBEDDING_DIR.mkdir(
    parents=True,
    exist_ok=True
)


np.save(
    EMBEDDING_DIR / "train_tweet_embeddings.npy",
    train_tweet_embeddings
)

np.save(
    EMBEDDING_DIR / "val_tweet_embeddings.npy",
    val_tweet_embeddings
)

np.save(
    EMBEDDING_DIR / "test_tweet_embeddings.npy",
    test_tweet_embeddings
)


print("=" * 90)
print("TWEET EMBEDDINGS SAVED")
print("=" * 90)

print(EMBEDDING_DIR)

print("\nFiles:")
for f in os.listdir(EMBEDDING_DIR):
    print(f)

TWEET EMBEDDINGS SAVED
C:\Users\p.vazifeh\Desktop\University\Project\Bot Detection Implementation\final_data\embeddings\xlmr

Files:
test_tweet_embeddings.npy
test_user_embeddings.npy
test_user_keys.npy
train_tweet_embeddings.npy
train_user_embeddings.npy
train_user_keys.npy
val_tweet_embeddings.npy
val_user_embeddings.npy
val_user_keys.npy


In [32]:
import numpy as np
import pandas as pd
import os


def build_user_embeddings(
    text_dataframe,
    tweet_embeddings
):

    assert len(text_dataframe) == len(tweet_embeddings), \
        "Text rows and embeddings rows do not match"


    temp_df = text_dataframe[
        [
            "user_key"
        ]
    ].copy()


    temp_df["embedding_idx"] = range(
        len(temp_df)
    )


    user_embeddings = []


    user_keys = []


    for user_key, group in temp_df.groupby(
        "user_key"
    ):

        indices = group["embedding_idx"].values


        user_vector = tweet_embeddings[
            indices
        ].mean(
            axis=0
        )


        user_embeddings.append(
            user_vector
        )


        user_keys.append(
            user_key
        )


    user_embeddings = np.vstack(
        user_embeddings
    )


    user_embedding_df = pd.DataFrame(
        {
            "user_key": user_keys
        }
    )


    return (
        user_embedding_df,
        user_embeddings
    )



print("User embedding aggregation function ready")

User embedding aggregation function ready


In [33]:
# Build user-level text embeddings

train_user_keys, train_user_embeddings = build_user_embeddings(
    train_text_labeled,
    train_tweet_embeddings
)


val_user_keys, val_user_embeddings = build_user_embeddings(
    val_text_labeled,
    val_tweet_embeddings
)


test_user_keys, test_user_embeddings = build_user_embeddings(
    test_text_labeled,
    test_tweet_embeddings
)


print("=" * 90)
print("USER LEVEL EMBEDDINGS")
print("=" * 90)


print(
    "Train:"
)
print(
    train_user_embeddings.shape
)


print(
    "\nValidation:"
)
print(
    val_user_embeddings.shape
)


print(
    "\nTest:"
)
print(
    test_user_embeddings.shape
)


print("\nUser counts:")

print(
    "Train users:",
    len(train_user_keys)
)

print(
    "Validation users:",
    len(val_user_keys)
)

print(
    "Test users:",
    len(test_user_keys)
)

USER LEVEL EMBEDDINGS
Train:
(671, 768)

Validation:
(144, 768)

Test:
(143, 768)

User counts:
Train users: 671
Validation users: 144
Test users: 143


In [34]:
# Save user-level XLM-R embeddings

USER_EMBEDDING_DIR = FINAL_DATA / "embeddings" / "xlmr"

USER_EMBEDDING_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# Save embeddings

np.save(
    USER_EMBEDDING_DIR / "train_user_embeddings.npy",
    train_user_embeddings
)

np.save(
    USER_EMBEDDING_DIR / "val_user_embeddings.npy",
    val_user_embeddings
)

np.save(
    USER_EMBEDDING_DIR / "test_user_embeddings.npy",
    test_user_embeddings
)


# Save user keys for alignment

np.save(
    USER_EMBEDDING_DIR / "train_user_keys.npy",
    np.array(train_user_keys)
)

np.save(
    USER_EMBEDDING_DIR / "val_user_keys.npy",
    np.array(val_user_keys)
)

np.save(
    USER_EMBEDDING_DIR / "test_user_keys.npy",
    np.array(test_user_keys)
)


print("=" * 90)
print("USER EMBEDDINGS SAVED")
print("=" * 90)

print(USER_EMBEDDING_DIR)

print("\nFiles:")

for file in sorted(
    os.listdir(USER_EMBEDDING_DIR)
):
    print(file)

USER EMBEDDINGS SAVED
C:\Users\p.vazifeh\Desktop\University\Project\Bot Detection Implementation\final_data\embeddings\xlmr

Files:
test_tweet_embeddings.npy
test_user_embeddings.npy
test_user_keys.npy
train_tweet_embeddings.npy
train_user_embeddings.npy
train_user_keys.npy
val_tweet_embeddings.npy
val_user_embeddings.npy
val_user_keys.npy


In [37]:
# Reload XLM-R user keys and fix shape

train_user_keys = np.load(
    USER_EMBEDDING_DIR / "train_user_keys.npy",
    allow_pickle=True
)

val_user_keys = np.load(
    USER_EMBEDDING_DIR / "val_user_keys.npy",
    allow_pickle=True
)

test_user_keys = np.load(
    USER_EMBEDDING_DIR / "test_user_keys.npy",
    allow_pickle=True
)


# Flatten if needed

train_user_keys = np.array(train_user_keys).flatten()
val_user_keys = np.array(val_user_keys).flatten()
test_user_keys = np.array(test_user_keys).flatten()


print("Shapes after flatten:")

print(train_user_keys.shape)
print(val_user_keys.shape)
print(test_user_keys.shape)


print("\nSamples:")
print(train_user_keys[:5])

Shapes after flatten:
(671,)
(144,)
(143,)

Samples:
['007mi00' '0mhp037' '2002kingfox' '2expvzwq7llpx4p' '2tosgzcfj8vtsjk']


In [38]:
def normalize_key(x):
    return str(x).strip().lower()


train_user_keys_norm = [
    normalize_key(k)
    for k in train_user_keys
]

val_user_keys_norm = [
    normalize_key(k)
    for k in val_user_keys
]

test_user_keys_norm = [
    normalize_key(k)
    for k in test_user_keys
]


train_label_map = dict(
    zip(
        train_tabular["user_key"].apply(normalize_key),
        train_tabular["label_binary"]
    )
)


val_label_map = dict(
    zip(
        val_tabular["user_key"].apply(normalize_key),
        val_tabular["label_binary"]
    )
)


test_label_map = dict(
    zip(
        test_tabular["user_key"].apply(normalize_key),
        test_tabular["label_binary"]
    )
)


def check_alignment(user_keys, label_map, name):

    labels = [
        label_map[k]
        for k in user_keys
        if k in label_map
    ]

    print("\n" + name)

    print("Users:", len(user_keys))
    print("Missing labels:", len(user_keys)-len(labels))

    print("Label distribution:")
    print(pd.Series(labels).value_counts())


check_alignment(
    train_user_keys_norm,
    train_label_map,
    "Train"
)


check_alignment(
    val_user_keys_norm,
    val_label_map,
    "Validation"
)


check_alignment(
    test_user_keys_norm,
    test_label_map,
    "Test"
)


Train
Users: 671
Missing labels: 0
Label distribution:
0    540
1    131
Name: count, dtype: int64

Validation
Users: 144
Missing labels: 0
Label distribution:
0    116
1     28
Name: count, dtype: int64

Test
Users: 143
Missing labels: 0
Label distribution:
0    115
1     28
Name: count, dtype: int64


In [39]:
# Build labels aligned with XLM-R user embeddings

def build_aligned_labels(
    user_keys,
    label_map
):

    labels = np.array(
        [
            label_map[
                str(k).strip().lower()
            ]
            for k in user_keys
        ]
    )

    return labels



y_train_text = build_aligned_labels(
    train_user_keys,
    train_label_map
)


y_val_text = build_aligned_labels(
    val_user_keys,
    val_label_map
)


y_test_text = build_aligned_labels(
    test_user_keys,
    test_label_map
)



print("=" * 90)
print("TEXT LABEL VECTORS")
print("=" * 90)


print("Train:")
print(
    train_user_embeddings.shape,
    y_train_text.shape
)


print("\nValidation:")
print(
    val_user_embeddings.shape,
    y_val_text.shape
)


print("\nTest:")
print(
    test_user_embeddings.shape,
    y_test_text.shape
)


print("\nLabel distribution:")

print("Train")
print(pd.Series(y_train_text).value_counts())

print("\nValidation")
print(pd.Series(y_val_text).value_counts())

print("\nTest")
print(pd.Series(y_test_text).value_counts())

TEXT LABEL VECTORS
Train:
(671, 768) (671,)

Validation:
(144, 768) (144,)

Test:
(143, 768) (143,)

Label distribution:
Train
0    540
1    131
Name: count, dtype: int64

Validation
0    116
1     28
Name: count, dtype: int64

Test
0    115
1     28
Name: count, dtype: int64


In [40]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)


# Train Text Logistic Regression

text_lr = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)


text_lr.fit(
    train_user_embeddings,
    y_train_text
)


print("Text Logistic Regression trained successfully")

Text Logistic Regression trained successfully


In [41]:
# Text Logistic Regression Evaluation

def evaluate_model(
    model,
    X,
    y,
    split_name
):

    y_prob = model.predict_proba(X)[:, 1]

    y_pred = (y_prob >= 0.5).astype(int)


    print("=" * 90)
    print(f"Text Logistic Regression - {split_name}")
    print("=" * 90)


    print(
        f"Accuracy:  {accuracy_score(y, y_pred):.4f}"
    )

    print(
        f"Precision: {precision_score(y, y_pred):.4f}"
    )

    print(
        f"Recall:    {recall_score(y, y_pred):.4f}"
    )

    print(
        f"F1:        {f1_score(y, y_pred):.4f}"
    )

    print(
        f"ROC_AUC:   {roc_auc_score(y, y_prob):.4f}"
    )

    print(
        f"PR_AUC:    {average_precision_score(y, y_prob):.4f}"
    )


    print("\nConfusion Matrix:")
    print(
        confusion_matrix(
            y,
            y_pred
        )
    )


    print("\nClassification Report:")
    print(
        classification_report(
            y,
            y_pred,
            target_names=[
                "Human",
                "Bot"
            ]
        )
    )


    return {
        "Accuracy": accuracy_score(y, y_pred),
        "Precision": precision_score(y, y_pred),
        "Recall": recall_score(y, y_pred),
        "F1": f1_score(y, y_pred),
        "ROC_AUC": roc_auc_score(y, y_prob),
        "PR_AUC": average_precision_score(y, y_prob)
    }



text_lr_val_results = evaluate_model(
    text_lr,
    val_user_embeddings,
    y_val_text,
    "Validation"
)


text_lr_test_results = evaluate_model(
    text_lr,
    test_user_embeddings,
    y_test_text,
    "Test"
)

Text Logistic Regression - Validation
Accuracy:  0.6181
Precision: 0.3043
Recall:    0.7500
F1:        0.4330
ROC_AUC:   0.7235
PR_AUC:    0.3696

Confusion Matrix:
[[68 48]
 [ 7 21]]

Classification Report:
              precision    recall  f1-score   support

       Human       0.91      0.59      0.71       116
         Bot       0.30      0.75      0.43        28

    accuracy                           0.62       144
   macro avg       0.61      0.67      0.57       144
weighted avg       0.79      0.62      0.66       144

Text Logistic Regression - Test
Accuracy:  0.5874
Precision: 0.2540
Recall:    0.5714
F1:        0.3516
ROC_AUC:   0.6562
PR_AUC:    0.3814

Confusion Matrix:
[[68 47]
 [12 16]]

Classification Report:
              precision    recall  f1-score   support

       Human       0.85      0.59      0.70       115
         Bot       0.25      0.57      0.35        28

    accuracy                           0.59       143
   macro avg       0.55      0.58      0.52  

In [42]:
import pandas as pd
import os


results_dir = os.path.join(
    PROJECT_ROOT,
    "results"
)

os.makedirs(
    results_dir,
    exist_ok=True
)


text_lr_results = pd.DataFrame(
    [
        {
            "Model": "Text Logistic Regression - Validation",
            **text_lr_val_results
        },
        {
            "Model": "Text Logistic Regression - Test",
            **text_lr_test_results
        }
    ]
)


text_lr_results = text_lr_results[
    [
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC_AUC",
        "PR_AUC"
    ]
]


text_lr_path = os.path.join(
    results_dir,
    "text_baseline_results.csv"
)


text_lr_results.to_csv(
    text_lr_path,
    index=False
)


print("=" * 90)
print("TEXT BASELINE RESULTS SAVED")
print("=" * 90)

print(text_lr_path)

text_lr_results

TEXT BASELINE RESULTS SAVED
C:\Users\p.vazifeh\Desktop\University\Project\Bot Detection Implementation\results\text_baseline_results.csv


,Model,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC
0,Text Logistic Regression - Validation,0.618056,0.304348,0.750000,0.432990,0.723522,0.369559
1,Text Logistic Regression - Test,0.587413,0.253968,0.571429,0.351648,0.656211,0.381394


In [43]:
import torch
import torch.nn as nn


class TextMLP(nn.Module):

    def __init__(self, input_dim=768):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(
                input_dim,
                256
            ),

            nn.BatchNorm1d(
                256
            ),

            nn.ReLU(),

            nn.Dropout(
                0.3
            ),


            nn.Linear(
                256,
                64
            ),

            nn.ReLU(),

            nn.Dropout(
                0.3
            ),


            nn.Linear(
                64,
                1
            )
        )


    def forward(self, x):

        return self.network(x)



text_mlp = TextMLP(
    input_dim=768
)


print("=" * 90)
print("TEXT MLP MODEL")
print("=" * 90)

print(text_mlp)

TEXT MLP MODEL
TextMLP(
  (network): Sequential(
    (0): Linear(in_features=768, out_features=256, bias=True)
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=256, out_features=64, bias=True)
    (5): ReLU()
    (6): Dropout(p=0.3, inplace=False)
    (7): Linear(in_features=64, out_features=1, bias=True)
  )
)


In [44]:
from torch.utils.data import TensorDataset, DataLoader


# =====================================
# Convert embeddings and labels to tensors
# =====================================

X_train_text_tensor = torch.tensor(
    train_user_embeddings,
    dtype=torch.float32
)

y_train_text_tensor = torch.tensor(
    y_train_text.reshape(-1, 1),
    dtype=torch.float32
)


X_val_text_tensor = torch.tensor(
    val_user_embeddings,
    dtype=torch.float32
)

y_val_text_tensor = torch.tensor(
    y_val_text.reshape(-1, 1),
    dtype=torch.float32
)


X_test_text_tensor = torch.tensor(
    test_user_embeddings,
    dtype=torch.float32
)

y_test_text_tensor = torch.tensor(
    y_test_text.reshape(-1, 1),
    dtype=torch.float32
)



# =====================================
# Dataset
# =====================================

train_text_dataset = TensorDataset(
    X_train_text_tensor,
    y_train_text_tensor
)


val_text_dataset = TensorDataset(
    X_val_text_tensor,
    y_val_text_tensor
)


test_text_dataset = TensorDataset(
    X_test_text_tensor,
    y_test_text_tensor
)



# =====================================
# DataLoader
# =====================================

train_text_loader = DataLoader(
    train_text_dataset,
    batch_size=32,
    shuffle=True
)


val_text_loader = DataLoader(
    val_text_dataset,
    batch_size=32,
    shuffle=False
)


test_text_loader = DataLoader(
    test_text_dataset,
    batch_size=32,
    shuffle=False
)



print("=" * 90)
print("TEXT MLP DATALOADERS")
print("=" * 90)

print("Train batches:", len(train_text_loader))
print("Validation batches:", len(val_text_loader))
print("Test batches:", len(test_text_loader))


print("\nTensor shapes:")
print("X_train:", X_train_text_tensor.shape)
print("y_train:", y_train_text_tensor.shape)

TEXT MLP DATALOADERS
Train batches: 21
Validation batches: 5
Test batches: 5

Tensor shapes:
X_train: torch.Size([671, 768])
y_train: torch.Size([671, 1])


In [45]:
import torch.optim as optim


# =====================================
# Device
# =====================================

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)


text_mlp = text_mlp.to(device)



# =====================================
# Class weight
# =====================================

num_positive = y_train_text.sum()
num_negative = len(y_train_text) - num_positive


pos_weight = torch.tensor(
    [
        num_negative / num_positive
    ],
    dtype=torch.float32
).to(device)



# =====================================
# Loss and Optimizer
# =====================================

criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)


optimizer = optim.AdamW(
    text_mlp.parameters(),
    lr=1e-3
)



print("=" * 90)
print("TEXT MLP TRAINING CONFIGURATION")
print("=" * 90)


print("Device:", device)

print(
    "Positive class weight:",
    pos_weight.item()
)

print(
    "Optimizer: AdamW"
)

print(
    "Learning rate: 1e-3"
)

print(
    "Loss: BCEWithLogitsLoss"
)

TEXT MLP TRAINING CONFIGURATION
Device: cpu
Positive class weight: 4.122137546539307
Optimizer: AdamW
Learning rate: 1e-3
Loss: BCEWithLogitsLoss


In [46]:
import numpy as np


EPOCHS = 50


train_losses = []
val_losses = []


best_val_loss = np.inf


print("=" * 90)
print("TRAINING TEXT MLP")
print("=" * 90)


for epoch in range(EPOCHS):

    # --------------------
    # Train
    # --------------------

    text_mlp.train()

    running_train_loss = 0.0


    for X_batch, y_batch in train_text_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)


        optimizer.zero_grad()


        logits = text_mlp(
            X_batch
        )


        loss = criterion(
            logits,
            y_batch
        )


        loss.backward()


        optimizer.step()


        running_train_loss += loss.item()



    train_loss = (
        running_train_loss /
        len(train_text_loader)
    )



    # --------------------
    # Validation
    # --------------------

    text_mlp.eval()

    running_val_loss = 0.0


    with torch.no_grad():

        for X_batch, y_batch in val_text_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)


            logits = text_mlp(
                X_batch
            )


            loss = criterion(
                logits,
                y_batch
            )


            running_val_loss += loss.item()



    val_loss = (
        running_val_loss /
        len(val_text_loader)
    )


    train_losses.append(train_loss)
    val_losses.append(val_loss)



    if val_loss < best_val_loss:

        best_val_loss = val_loss

        best_model_state = text_mlp.state_dict()



    if (epoch + 1) % 5 == 0:

        print(
            f"Epoch [{epoch+1}/{EPOCHS}] "
            f"Train Loss: {train_loss:.4f} "
            f"Val Loss: {val_loss:.4f}"
        )


print("\nTraining completed")

TRAINING TEXT MLP
Epoch [5/50] Train Loss: 0.8401 Val Loss: 1.1418
Epoch [10/50] Train Loss: 0.5780 Val Loss: 1.5723
Epoch [15/50] Train Loss: 0.4266 Val Loss: 1.4997
Epoch [20/50] Train Loss: 0.4336 Val Loss: 3.2407
Epoch [25/50] Train Loss: 0.2772 Val Loss: 2.2279
Epoch [30/50] Train Loss: 0.2209 Val Loss: 2.4945
Epoch [35/50] Train Loss: 0.2102 Val Loss: 3.3933
Epoch [40/50] Train Loss: 0.1592 Val Loss: 5.0366
Epoch [45/50] Train Loss: 0.2556 Val Loss: 2.7435
Epoch [50/50] Train Loss: 0.1438 Val Loss: 3.5111

Training completed


In [47]:
# Restore best validation model

text_mlp.load_state_dict(
    best_model_state
)

print("Best Text MLP model restored")

Best Text MLP model restored


In [48]:
import numpy as np
import torch


def predict_text_mlp(model, loader):

    model.eval()

    probabilities = []


    with torch.no_grad():

        for X_batch, _ in loader:

            X_batch = X_batch.to(device)

            logits = model(
                X_batch
            )

            probs = torch.sigmoid(
                logits
            )

            probabilities.extend(
                probs.cpu().numpy().flatten()
            )


    return np.array(probabilities)



val_prob_text_mlp = predict_text_mlp(
    text_mlp,
    val_text_loader
)


test_prob_text_mlp = predict_text_mlp(
    text_mlp,
    test_text_loader
)


print("=" * 90)
print("TEXT MLP PREDICTION")
print("=" * 90)


print(
    "Validation probabilities:",
    val_prob_text_mlp.shape
)

print(
    "Test probabilities:",
    test_prob_text_mlp.shape
)

TEXT MLP PREDICTION
Validation probabilities: (144,)
Test probabilities: (143,)


In [50]:
text_mlp_val_results = evaluate_predictions(
    y_val_text,
    val_prob_text_mlp,
    "Text MLP - Validation"
)


text_mlp_test_results = evaluate_predictions(
    y_test_text,
    test_prob_text_mlp,
    "Text MLP - Test"
)

Text MLP - Validation
Accuracy:  0.6736
Precision: 0.2121
Recall:    0.2500
F1:        0.2295
ROC_AUC:   0.6053
PR_AUC:    0.2580

Confusion Matrix:
[[90 26]
 [21  7]]

Classification Report:
              precision    recall  f1-score   support

       Human       0.81      0.78      0.79       116
         Bot       0.21      0.25      0.23        28

    accuracy                           0.67       144
   macro avg       0.51      0.51      0.51       144
weighted avg       0.69      0.67      0.68       144

Text MLP - Test
Accuracy:  0.7343
Precision: 0.2500
Recall:    0.1786
F1:        0.2083
ROC_AUC:   0.5910
PR_AUC:    0.2764

Confusion Matrix:
[[100  15]
 [ 23   5]]

Classification Report:
              precision    recall  f1-score   support

       Human       0.81      0.87      0.84       115
         Bot       0.25      0.18      0.21        28

    accuracy                           0.73       143
   macro avg       0.53      0.52      0.52       143
weighted avg       

In [51]:
text_mlp_val_results = {
    "Accuracy": 0.6736,
    "Precision": 0.2121,
    "Recall": 0.2500,
    "F1": 0.2295,
    "ROC_AUC": 0.6053,
    "PR_AUC": 0.2580
}


text_mlp_test_results = {
    "Accuracy": 0.7343,
    "Precision": 0.2500,
    "Recall": 0.1786,
    "F1": 0.2083,
    "ROC_AUC": 0.5910,
    "PR_AUC": 0.2764
}